# Интелигентна система за управление на поливане чрез размита логика

## 1. Мотивация на задачата

Поливането на растенията зависи от няколко фактора, които често не могат да се оценят напълно точно. Например почвата може да бъде „леко влажна“, температурата може да бъде „умерено висока“, а влажността на въздуха може да бъде „ниска“. Това са неточни и приблизителни понятия, които са подходящи за моделиране чрез размита логика.

Целта на проекта е да се разработи интелигентна система, която определя препоръчителното количество вода за поливане според текущите условия. Вместо да се използват строги граници от типа „ако температурата е над 30°C, поливай много“, системата работи с размити множества и правила, близки до човешкото мислене.

## 2. Описание на задачата

Разработва се fuzzy control система, която приема три входни величини:

- **Влажност на почвата** – показва колко суха или влажна е почвата;
- **Температура на въздуха** – показва дали условията са студени, нормални или горещи;
- **Влажност на въздуха** – показва дали въздухът е сух, нормален или влажен.

Изходната величина е:

- **Количество вода за поливане** – ниско, средно или високо количество вода.

Системата използва размити правила от типа:

> Ако почвата е суха и температурата е висока, тогава количеството вода трябва да бъде високо.

## 3. Имплементация на задачата

Проектът е реализиран с Python и библиотеката `scikit-fuzzy`. Използвани са триъгълни и трапецовидни функции на принадлежност. След дефиниране на входните и изходната променлива се създава база от правила. За дефазификация се използва centroid методът, който преобразува размития резултат в конкретна числова стойност.



In [ ]:
# Ако работите в Google Colab, първо изпълнете този ред:
# !pip install scikit-fuzzy


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt


## 4. Дефиниране на входни и изходни променливи

В проекта се използват следните универсални множества:

- влажност на почвата: от 0 до 100%;
- температура: от 0 до 45°C;
- влажност на въздуха: от 0 до 100%;
- количество вода: от 0 до 10 литра.



In [ ]:
soil_moisture = ctrl.Antecedent(np.arange(0, 101, 1), 'soil_moisture')
temperature = ctrl.Antecedent(np.arange(0, 46, 1), 'temperature')
air_humidity = ctrl.Antecedent(np.arange(0, 101, 1), 'air_humidity')

water_amount = ctrl.Consequent(np.arange(0, 11, 1), 'water_amount')


## 5. Функции на принадлежност

За всяка входна величина са зададени по три лингвистични стойности. Например влажността на почвата може да бъде **суха**, **нормална** или **влажна**. По същия начин температурата може да бъде **ниска**, **умерена** или **висока**.



In [ ]:
# Влажност на почвата
soil_moisture['dry'] = fuzz.trapmf(soil_moisture.universe, [0, 0, 20, 40])
soil_moisture['normal'] = fuzz.trimf(soil_moisture.universe, [30, 50, 70])
soil_moisture['wet'] = fuzz.trapmf(soil_moisture.universe, [60, 80, 100, 100])

# Температура
temperature['low'] = fuzz.trapmf(temperature.universe, [0, 0, 10, 18])
temperature['medium'] = fuzz.trimf(temperature.universe, [15, 23, 31])
temperature['high'] = fuzz.trapmf(temperature.universe, [28, 35, 45, 45])

# Влажност на въздуха
air_humidity['low'] = fuzz.trapmf(air_humidity.universe, [0, 0, 25, 45])
air_humidity['medium'] = fuzz.trimf(air_humidity.universe, [35, 55, 75])
air_humidity['high'] = fuzz.trapmf(air_humidity.universe, [65, 80, 100, 100])

# Количество вода
water_amount['low'] = fuzz.trapmf(water_amount.universe, [0, 0, 2, 4])
water_amount['medium'] = fuzz.trimf(water_amount.universe, [3, 5, 7])
water_amount['high'] = fuzz.trapmf(water_amount.universe, [6, 8, 10, 10])


In [ ]:
soil_moisture.view()
temperature.view()
air_humidity.view()
water_amount.view()
plt.show()


## 6. База от размити правила

Правилата описват логиката на системата. Те са формулирани така, че да наподобяват човешко разсъждение при вземане на решение за поливане.



In [ ]:
rule1 = ctrl.Rule(soil_moisture['dry'] & temperature['high'], water_amount['high'])
rule2 = ctrl.Rule(soil_moisture['dry'] & temperature['medium'], water_amount['high'])
rule3 = ctrl.Rule(soil_moisture['dry'] & air_humidity['low'], water_amount['high'])

rule4 = ctrl.Rule(soil_moisture['normal'] & temperature['high'] & air_humidity['low'], water_amount['medium'])
rule5 = ctrl.Rule(soil_moisture['normal'] & temperature['medium'], water_amount['medium'])
rule6 = ctrl.Rule(soil_moisture['normal'] & air_humidity['high'], water_amount['low'])

rule7 = ctrl.Rule(soil_moisture['wet'], water_amount['low'])
rule8 = ctrl.Rule(temperature['low'] & air_humidity['high'], water_amount['low'])
rule9 = ctrl.Rule(soil_moisture['normal'] & temperature['low'], water_amount['low'])

rule10 = ctrl.Rule(soil_moisture['dry'] & temperature['low'], water_amount['medium'])
rule11 = ctrl.Rule(soil_moisture['wet'] & temperature['high'], water_amount['medium'])


## 7. Създаване на контролна система

След дефиниране на входовете, изхода и правилата се създава fuzzy control система. Тя може да получава конкретни стойности и да изчислява препоръчително количество вода.



In [ ]:
irrigation_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3, rule4, rule5, rule6,
    rule7, rule8, rule9, rule10, rule11
])

irrigation_simulation = ctrl.ControlSystemSimulation(irrigation_ctrl)


## 8. Примерно тестване на системата

В този пример се разглежда ситуация, при която:

- почвата е доста суха – 25%;
- температурата е висока – 32°C;
- влажността на въздуха е ниска – 30%.

Очакването е системата да препоръча по-голямо количество вода.



In [ ]:
irrigation_simulation.input['soil_moisture'] = 25
irrigation_simulation.input['temperature'] = 32
irrigation_simulation.input['air_humidity'] = 30

irrigation_simulation.compute()

result = irrigation_simulation.output['water_amount']
print(f"Препоръчително количество вода: {result:.2f} литра")

water_amount.view(sim=irrigation_simulation)
plt.show()


## 9. Функция за повторно използване

Следващата функция позволява лесно тестване с различни входни стойности.



In [ ]:
def evaluate_irrigation(soil, temp, humidity):
    simulation = ctrl.ControlSystemSimulation(irrigation_ctrl)

    simulation.input['soil_moisture'] = soil
    simulation.input['temperature'] = temp
    simulation.input['air_humidity'] = humidity

    simulation.compute()

    soil_moisture.view(sim=simulation)
    temperature.view(sim=simulation)
    air_humidity.view(sim=simulation)
    water_amount.view(sim=simulation)

    plt.show()

    water = simulation.output['water_amount']

    if water < 4:
        label = "ниско количество вода"
    elif water < 7:
        label = "средно количество вода"
    else:
        label = "високо количество вода"

    return water, label


test_cases = [
    (20, 35, 25),
    (50, 24, 55),
    (85, 20, 70),
    (40, 38, 30),
    (65, 32, 35)
]

for soil, temp, humidity in test_cases:
    water, label = evaluate_irrigation(soil, temp, humidity)
    print(f"Почва: {soil}% | Температура: {temp}°C | Влажност: {humidity}% -> {water:.2f} л. ({label})")


## 10. Заключение

Разработената система демонстрира как размитата логика може да се използва за вземане на решения при условия на неопределеност. Вместо да работи с твърди граници, системата използва лингвистични стойности като „суха почва“, „висока температура“ и „ниска влажност“. Това прави модела по-гъвкав и по-близък до реалното човешко мислене.

Проектът може да бъде разширен чрез добавяне на още входни величини, например сезон, тип растение, прогноза за дъжд или час от деня. Така системата би могла да стане по-точна и приложима в реална автоматизирана поливна система.
